# Cross-task remapping + coherent rotation (PFC, mFC dataset)

El-Gaby et al. 2024 (Figure 3) reimplemented on the mPFC dataset via [`remapping_rotation_analysis.py`](remapping_rotation_analysis.py): cross-task remapping of task-state (goal-progress) tuning, coherent rotation of neuron pairs, and the module-clustering / silhouette analysis. Running on mPFC also validates the reimplementation against the published result.

Only the data-loading cells are PFC-specific (the PFC loader builds the LEC-shaped `data_dic` from disk). The analysis works from raw fields (`Neuron_raw` / `Trial_times` / `Task`), so `compute_norm=False` is sufficient.

In [1]:
import numpy as np
import scipy.stats as st
from scipy import stats
from scipy.stats import zscore
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os, pickle
from tqdm import tqdm

In [2]:
DATA_FOLDER = '/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data'
META = os.path.join(DATA_FOLDER, 'MetaData')

In [3]:
# Load the canonical recday list (25 double-day recordings, ABCD tasks only)
mouse_recdays = list(np.load(os.path.join(META, 'combined_ABCDonly_days.npy')).astype(str))
print(f'{len(mouse_recdays)} recdays in combined_ABCDonly_days.npy')
print('first 3:', mouse_recdays[:3])

25 recdays in combined_ABCDonly_days.npy
first 3: [np.str_('ab03_01092023_02092023'), np.str_('ab03_05092023_06092023'), np.str_('ab03_29082023_30082023')]


In [4]:
# Build the data_dic from the PFC directory layout. The remapping/rotation analysis only needs
# raw fields (Neuron_raw / Trial_times / Task), so compute_norm=False keeps it light (no extra
# Neurons_norm/Locs_norm). For a quick first pass, uncomment the next line to slice down.
# mouse_recdays = mouse_recdays[:5]

from glm_analysis_v2 import build_data_dic_from_pfc
data_dic = build_data_dic_from_pfc(DATA_FOLDER, mouse_recdays, compute_norm=False)
mouse_recdays = sorted(data_dic.keys())   # in case any were skipped
print(f'\n{len(mouse_recdays)} recdays loaded (raw fields: Neuron_raw / Locs_raw / Trial_times / Task)')

  ab03_01092023_02092023: 7 sessions
  ab03_05092023_06092023: 8 sessions
  ab03_29082023_30082023: 9 sessions
  ah03_12082021_13082021: 8 sessions
  ah03_18082021_19082021: 8 sessions
  ah04_01122021_02122021: 8 sessions
  ah04_05122021_06122021: 8 sessions
  ah04_07122021_08122021: 8 sessions
  ah04_09122021_10122021: 8 sessions
  ah04_14122021_16122021: 8 sessions
  ah07_01092023_02092023: 7 sessions
  ah07_27082023_28082023: 7 sessions
  ah07_29082023_30082023: 7 sessions
  me08_06092021_09092021: 8 sessions
  me08_10092021_11092021: 6 sessions
  me08_12092021_13092021: 7 sessions
  me10_09122021_10122021: 9 sessions
  me10_14122021_15122021: 6 sessions
  me10_17122021_19122021: 8 sessions
  me10_20122021_21122021: 6 sessions
  me11_01122021_02122021: 8 sessions
  me11_05122021_06122021: 9 sessions
  me11_07122021_08122021: 8 sessions
  me11_09122021_10122021: 9 sessions
  me11_12122021_13122021: 9 sessions

25 recdays loaded (raw fields: Neuron_raw / Locs_raw / Trial_times / Task)

In [5]:
# Build valid_sessions_dic: for each mouse_recday, keep one session per unique task structure.
# Identical logic to the LEC notebook; works on PFC because num_trials and Task are present.
valid_sessions_dic = {}
for mouse_recday in mouse_recdays:
    valid_sessions = []
    tasks = []
    for session in list(data_dic[mouse_recday].keys()):
        if session == 'valid_sessions':
            continue
        if data_dic[mouse_recday][session]['num_trials'] < 5:
            print(f'{mouse_recday} session {session}: not enough trials, skipping')
            continue
        if 'defaultdict' in str(data_dic[mouse_recday][session]['Task']):
            print(f'{mouse_recday} session {session}: no task, skipping')
            continue
        if not any(np.array_equal(data_dic[mouse_recday][session]['Task'], candidate)
                   for candidate in tasks):
            tasks.append(data_dic[mouse_recday][session]['Task'])
            valid_sessions.append(session)
    print(f'{mouse_recday}: {valid_sessions}')
    valid_sessions_dic[mouse_recday] = valid_sessions

ab03_01092023_02092023: [0, 1, 2, 3, 4, 5]
ab03_05092023_06092023: [0, 1, 2, 4, 5, 6]
ab03_29082023_30082023: [0, 1, 2, 4, 6, 7]
ah03_12082021_13082021: [0, 1, 2, 4, 5, 6]
ah03_18082021_19082021: [0, 1, 2, 4, 5, 6]
ah04_01122021_02122021: [0, 1, 2, 4, 5, 6]
ah04_05122021_06122021: [0, 1, 2, 4, 5, 6]
ah04_07122021_08122021 session 2: not enough trials, skipping
ah04_07122021_08122021: [0, 1, 3, 4, 5, 6]
ah04_09122021_10122021: [0, 1, 2, 4, 5, 6]
ah04_14122021_16122021: [0, 1, 2, 4, 5, 6]
ah07_01092023_02092023: [0, 1, 2, 3, 4, 5]
ah07_27082023_28082023: [0, 1, 2, 4, 5, 6]
ah07_29082023_30082023 session 3: not enough trials, skipping
ah07_29082023_30082023: [0, 1, 2, 4, 5, 6]
me08_06092021_09092021 session 2: not enough trials, skipping
me08_06092021_09092021: [0, 1, 3, 4, 5, 6]
me08_10092021_11092021: [0, 1, 2, 4, 5]
me08_12092021_13092021: [0, 1, 3, 4, 5, 6]
me10_09122021_10122021: [0, 1, 2, 4, 6, 7]
me10_14122021_15122021: [0, 1, 2, 4]
me10_17122021_19122021: [0, 1, 2, 4, 5, 6]
me10_2

# Cross-task remapping + coherent rotation (El-Gaby Figure3 reimplementation)

Uses `remapping_rotation_analysis.py`. For each recday it builds per-neuron, per-task 360-bin
goal-progress tuning curves (matched cells across tasks), then:
 * **remapping** — single-cell rotation offset vs a reference task (peak at 0 = generalises);
 * **coherent rotation** — whether neuron *pairs* keep their relative rotation across tasks
   (population rotates as a rigid body), vs a 1/num_states chance level;
 * **X-vs-X′** — the same on repeated identical-task sessions (no-remapping baseline).

Neurons are included if state-tuned in ≥half of their tasks. Figures saved to a separate
`remapping_rotation_<timestamp>` directory; v2/v3 outputs untouched.


In [6]:
exec(open('remapping_rotation_analysis.py').read())
from datetime import datetime

config_rr = RemapConfig()   # num_states=4, sigma=10, coherence threshold 45 deg, tuned in >=half tasks
save_dir_rr = f'../data/figures/remapping_rotation_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

results_rr, summary_rr = run_remapping_rotation_all_mice(
    data_dic, valid_sessions_dic, config=config_rr, save_dir=save_dir_rr, verbose=True)

print('\nSUMMARY:')
for k, v in summary_rr.items():
    print(f'  {k}: {v}')


Processing ab03_01092023_02092023 ...
  ab03_01092023_02092023: 6 tasks, 50 included neurons, coherent_prop=0.072
Processing ab03_05092023_06092023 ...
  ab03_05092023_06092023: 6 tasks, 25 included neurons, coherent_prop=0.163, X-vs-X' present
Processing ab03_29082023_30082023 ...
  ab03_29082023_30082023: 6 tasks, 42 included neurons, coherent_prop=0.142, X-vs-X' present
Processing ah03_12082021_13082021 ...
  ah03_12082021_13082021: 6 tasks, 9 included neurons, coherent_prop=nan, X-vs-X' present
Processing ah03_18082021_19082021 ...
  ah03_18082021_19082021: 6 tasks, 11 included neurons, coherent_prop=0.055, X-vs-X' present
Processing ah04_01122021_02122021 ...
  ah04_01122021_02122021: 6 tasks, 39 included neurons, coherent_prop=0.120, X-vs-X' present
Processing ah04_05122021_06122021 ...
  ah04_05122021_06122021: 6 tasks, 74 included neurons, coherent_prop=0.084, X-vs-X' present
Processing ah04_07122021_08122021 ...
  ah04_07122021_08122021: 6 tasks, 34 included neurons, coherent_

In [7]:
# Single recday (quick inspection)
mr = [m for m in data_dic if len(valid_sessions_dic.get(m, [])) >= 3][0]
res = analyse_recday(data_dic, mr, valid_sessions_dic[mr], config_rr, verbose=True)
print(mr, 'included neurons:', res['n_included'],
      '| coherent_prop:', res.get('coherent_prop'))


  ab03_01092023_02092023: 6 tasks, 50 included neurons, coherent_prop=0.072
ab03_01092023_02092023 included neurons: 50 | coherent_prop: 0.07183673469387755
